# FractalX Quick Demo

This notebook generates a synthetic fractal-like user history, estimates intrinsic dimension, scores a good candidate post and a garbage candidate post, and plots the 2D geometry.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "prototype.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from prototype import FractalInterferenceScorer, estimate_intrinsic_dimension_twonn

In [ ]:
def sierpinski_points(n_points=1200, seed=7):
    """Generate a simple Sierpinski-triangle point cloud."""
    rng = np.random.default_rng(seed)
    vertices = np.array([
        [0.0, 0.0],
        [1.0, 0.0],
        [0.5, np.sqrt(3.0) / 2.0],
    ])
    point = np.array([0.2, 0.1])
    points = []
    for _ in range(n_points + 20):
        vertex = vertices[rng.integers(0, len(vertices))]
        point = 0.5 * (point + vertex)
        points.append(point.copy())
    return np.array(points[20:])


history = sierpinski_points()
engagement_weights = np.ones(history.shape[0])

# A good post lands on the user's fractal-like interest support; a garbage post lands in the central hole.
good_post = np.array([[0.90, 0.05]])
garbage_post = np.array([[0.50, 0.30]])
candidates = np.vstack([good_post, garbage_post])

dimension = estimate_intrinsic_dimension_twonn(history, random_state=7)
scorer = FractalInterferenceScorer(
    smax=4,
    lambda0=0.50,
    wave_mode="random",
    n_wave_vectors=16,
    random_state=7,
).fit(history, engagement_weights)
scores = scorer.score(candidates)

print(f"Estimated intrinsic dimension: {dimension:.3f}")
print(f"Dimension ratio used by scorer: {scorer.dimension_ratio_:.3f}")
print(f"Scale weights: {np.round(scorer.scale_weights_, 3)}")
print(f"Good post score: {scores[0]:.4f}")
print(f"Garbage post score: {scores[1]:.4f}")

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(history[:, 0], history[:, 1], s=4, alpha=0.35, label="Synthetic user history")
plt.scatter(good_post[:, 0], good_post[:, 1], s=120, marker="*", label=f"Good post: {scores[0]:.2f}")
plt.scatter(garbage_post[:, 0], garbage_post[:, 1], s=120, marker="X", label=f"Garbage post: {scores[1]:.2f}")
plt.title("FractalX Synthetic User Interest Demo")
plt.xlabel("Embedding dimension 1")
plt.ylabel("Embedding dimension 2")
plt.axis("equal")
plt.grid(alpha=0.25)
plt.legend(loc="upper right")
plt.show()